# 07 — Deduplicación y fusión con decisiones revisadas

Cierre de la fase: **obra → manifestación bibliográfica → autor UNAM**.
Utiliza el plan de identidades y las decisiones completas de la revisión, no las
reglas antiguas de separación por `Fuente_origen`.

**Entradas (solo lectura):**
- `../04_Limpieza/03_limpieza_bibliografica/autores_unam_limpios.csv`
- `../04_Limpieza/04_deduplicacion/plan_manifestaciones.csv`
- `../04_Limpieza/04_deduplicacion/casos_revision_deduplicacion_resueltos.csv`

**Salidas en `../04_Limpieza/04_deduplicacion/`:**
- `autores_unam_deduplicados.csv`: 14 columnas, un índice por manifestación y una fila por autor.
- `auditoria_fusion.csv`: linaje de todas las filas e índices originales.
- `auditoria_campos_fusion.csv`: valores seleccionados, donantes y decisiones por campo.
- `resumen_fusion.csv`: conteos, huellas, limitaciones y excepciones.

Se conservan las 149 decisiones previas, se cierran las 476 pendientes y se
documenta una corrección conservadora adicional de especificidad institucional.
La revisión utilizó fuentes externas cuando fue necesario; **este notebook no
consulta Internet** y no completa campos desde fuentes externas.

La errata `10.1145/3712255.373426` → `10.1145/3712255.3734260` es una excepción
documentada exclusivamente para la nota *Hot off the Press*. No se aplica una
regla general que fusione DOI por distancia de un carácter. La nota permanece
separada del artículo de investigación al que hace referencia.

Excepciones de selección registradas explícitamente:
- Un año IMAV pasa de 2025 a 2024 tras contrastar la publicación.
- Cinco abstracts heredados no literales se dejan vacíos; todos sus valores
  anteriores permanecen en las decisiones y auditorías, para completado posterior.
- Un abstract conserva exactamente el fragmento anterior a la sección `Highlights`.
- Para una autora se conserva UNAM genérica ante evidencia contradictoria sobre
  una dependencia específica; no se le copia la adscripción de un coautor.

Los valores combinados/propagados deben tener donantes internos, dentro de la
misma manifestación o de una obra cuya armonización fue aprobada. No se añaden
autores, no se cambian sus nombres y no se fusionan DOI distintos fuera de la
errata expresamente resuelta. Los nuevos índices siguen el orden de primera
aparición de cada manifestación en la entrada.

`actualizar_archivos = False` protege salidas anteriores diferentes. Resguardar
la versión previa en GitHub Desktop antes de autorizar su reemplazo con `True`.
No modificar las huellas ni omitir validaciones para forzar una entrada distinta.

**Un cierre sin decisiones pendientes no significa que todos los campos estén
completos ni que toda incertidumbre científica haya desaparecido. Consultar
`Limitacion` en las decisiones y `resumen_fusion.csv`.**


In [1]:
# ============================================================
# 07 - CIERRE REVISADO: OBRA -> MANIFESTACIÓN -> AUTOR UNAM
# No recalibra, no consulta Internet y no reabre el código 06.
# ============================================================
import csv
import hashlib
import json
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit, unquote

import pandas as pd

# ============================================================
# 1. CONFIGURACIÓN - MISMAS CARPETAS DEL PROYECTO
# ============================================================
RAIZ_MANUAL = None
actualizar_archivos = False      # Resguardar la versión anterior en Git antes de cambiar a True.

SHA256_ENTRADA = 'e08804c671590b4f647ee57cbfde0a41a85995cc3ea3969885623274f81478e1'
SHA256_PLAN = '7f5e24a46cbda18775b061052fe78b3d97165dec85d3e932f9c23ab33ae77ab5'
SHA256_DECISIONES = '0f74ae283c6fbb4297a055d38e3b5072f1f2156d3b17984966f00a1498ddc341'
# Estas huellas fijan el checkpoint aprobado. No borrar comprobaciones para forzar otra entrada.

CANON = ['indice','Titulo','Año','Autor_norm','Afiliacion1','Afiliacion2',
         'ISBN','ISSN','Doi','URL','Area','SubArea','Keywords','Abstract']
BIB = ['Titulo','Año','ISBN','ISSN','Doi','URL','Area','SubArea','Keywords','Abstract']
AREAS = {'CC','IA','ISBD','RS','SIAV','TC'}
GENERICA = 'Universidad Nacional Autónoma de México'
SCI = re.compile(r'^[+-]?\d+(?:\.\d+)?[eE][+-]?\d+$')
DOI_RE = re.compile(r'^10\.\d{4,9}/[^\s<>]+$')
EXCEPCION_DOI = {'10.1145/3712255.373426': '10.1145/3712255.3734260'}
CASOS_RECHAZO_ABSTRACT = {
    'C_E8BCB0A410A177C3', 'C_1914265EDAD87ECB', 'C_D1A4371B6B248DDE',
    'C_C8C4759F90348F36', 'C_F60C20D222ABBB14'
}

# ============================================================
# 2. FUNCIONES DE LECTURA Y COMPARACIÓN (NO REESCRIBEN DATOS)
# ============================================================
def js(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def unicos(values):
    return list(dict.fromkeys(v for v in values if v != ''))


def ids(text):
    if not text.strip():
        return []
    out = [int(v.strip()) for v in text.split('|') if v.strip()]
    if any(v <= 0 for v in out) or len(out) != len(set(out)):
        raise ValueError('row_ids inválidos o repetidos: ' + text[:100])
    return out


def norm_text(text):
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFC', text)).strip()


def norm_title(text):
    text = norm_text(text).casefold()
    return ' '.join(''.join(c if c.isalnum() or unicodedata.category(c).startswith('S')
                            else ' ' for c in text).split())


def norm_id(text):
    return re.sub(r'[-\s]', '', text).upper()


def tokens(text):
    return [x.strip() for x in text.split(';') if x.strip()]


def union_tokens(values, campo):
    key = norm_id if campo in {'ISBN','ISSN'} else lambda x: norm_text(x).casefold()
    seen, result = set(), []
    for value in values:
        for token in tokens(value):
            k = key(token)
            if k not in seen:
                seen.add(k)
                result.append(token)
    return '; '.join(result)


def doi_url(url):
    p = urlsplit(url)
    if (p.hostname or '').lower() in {'doi.org','www.doi.org','dx.doi.org'}:
        return unquote(p.path).strip('/').casefold()
    return ''


def norm_url(url):
    if not url:
        return ''
    d = doi_url(url)
    if d:
        return 'https://doi.org/' + d
    p = urlsplit(url)
    return urlunsplit((p.scheme.lower(), p.netloc.lower(), p.path.rstrip('/'), p.query, ''))


def isbn_valido(token):
    s = norm_id(token)
    if re.fullmatch(r'\d{9}[\dX]', s):
        return sum((10-i) * (10 if c == 'X' else int(c)) for i,c in enumerate(s)) % 11 == 0
    if re.fullmatch(r'97[89]\d{10}', s):
        return sum((1 if i % 2 == 0 else 3) * int(c) for i,c in enumerate(s)) % 10 == 0
    return False


def issn_valido(token):
    if not re.fullmatch(r'\d{4}-\d{3}[\dX]', token):
        return False
    s = token.replace('-', '')
    return sum((8-i) * (10 if c == 'X' else int(c)) for i,c in enumerate(s)) % 11 == 0


def validar_identificadores(tabla):
    for campo, valid in [('ISBN',isbn_valido),('ISSN',issn_valido)]:
        for pos,value in enumerate(tabla[campo],1):
            for token in tokens(value):
                if SCI.fullmatch(token) or not valid(token):
                    raise ValueError(f'{campo} inválido, registro de datos {pos}: {token!r}')
    for value in tabla.Doi:
        if value and (not DOI_RE.fullmatch(value) or value != value.lower()):
            raise ValueError('DOI no limpio: ' + value)
    for value in tabla.URL:
        if value:
            p = urlsplit(value)
            if p.scheme not in {'http','https'} or not p.hostname or re.search(r'\s',value):
                raise ValueError('URL sintácticamente inválida: ' + value)


def leer_csv(path):
    path = Path(path)
    with path.open('rb') as f:
        if f.read(100).startswith(b'version https://git-lfs.github.com/spec/v1'):
            raise ValueError(f'{path.name} es un puntero Git LFS, no contiene los datos.')
    table = pd.read_csv(path,dtype=str,keep_default_na=False,encoding='utf-8-sig')
    if any(c.startswith('Unnamed') for c in table.columns):
        raise ValueError(f'{path.name}: columnas Unnamed; no se eliminan silenciosamente.')
    return table


def localizar_raiz():
    if RAIZ_MANUAL is not None:
        return Path(RAIZ_MANUAL).expanduser().resolve()
    inicio = Path.cwd().resolve()
    for p in [inicio, *inicio.parents]:
        if (p / '04_Limpieza').is_dir() and (p / 'notebooks').is_dir():
            return p
    raise FileNotFoundError('Ejecutar desde notebooks/ de Tesis_Multimodelo o indicar RAIZ_MANUAL.')


# ============================================================
# 3. VERIFICAR ENTRADA, PLAN Y DECISIONES CERRADAS
# ============================================================
def cargar_checkpoint(raiz):
    entrada_path = raiz / '04_Limpieza/03_limpieza_bibliografica/autores_unam_limpios.csv'
    carpeta = raiz / '04_Limpieza/04_deduplicacion'
    plan_path = carpeta / 'plan_manifestaciones.csv'
    revisiones_path = carpeta / 'casos_revision_deduplicacion_resueltos.csv'
    paths = [entrada_path,plan_path,revisiones_path]
    expected = [SHA256_ENTRADA,SHA256_PLAN,SHA256_DECISIONES]
    for path,h in zip(paths,expected):
        if sha256(path) != h:
            raise ValueError(f'{path.name}: la huella no coincide con el checkpoint revisado. No continuar ni regenerar con 06.')
    original,plan,rev = [leer_csv(p) for p in paths]
    if list(original.columns) != ['Fuente_origen'] + CANON:
        raise ValueError('La entrada debe conservar sus 15 columnas originales, en orden.')
    if original.shape != (5106,15):
        raise ValueError('No es la entrada de 5106 registros revisada.')
    validar_identificadores(original)
    if not original.Año.isin(['','2024','2025']).all() or not original.Area.isin(AREAS).all():
        raise ValueError('Año o Area fuera del esquema.')
    if original.SubArea.ne('').any() or original.Autor_norm.eq('').any():
        raise ValueError('SubArea con contenido o autor vacío.')
    if len(plan) != 970 or plan.Representacion_ID.duplicated().any():
        raise ValueError('El plan no contiene las 970 representaciones revisadas.')
    if len(rev) != 626 or rev.Caso_ID.duplicated().any():
        raise ValueError('Se esperan 626 decisiones: 149 previas + 476 resueltas + 1 corrección contextual.')
    if rev.Decision_manual.str.strip().eq('').any() or rev.Estado_cierre.str.strip().eq('').any():
        raise ValueError('Persisten decisiones sin cerrar.')
    if not plan.SHA256_entrada.eq(SHA256_ENTRADA).all() or not rev.SHA256_entrada.eq(SHA256_ENTRADA).all():
        raise ValueError('Plan/decisiones de otra entrada.')
    # Fuente_origen permanece solamente en original, para linaje. No se usa como frontera ni preferencia.
    work = original[CANON].copy(deep=True)
    return original,work,plan,rev,carpeta,paths,expected


def organizar_plan(work,plan):
    rows_m,rows_w,work_m,rep_m,score_m = defaultdict(list),defaultdict(list),{}, {}, {}
    m_row,w_row = {},{}
    for r in plan.to_dict('records'):
        ri = ids(r['row_ids_originales'])
        if not ri or max(ri)>len(work):
            raise ValueError('Referencia de filas fuera de la entrada.')
        w = r['Obra_manual'] or r['Obra_propuesta']
        m = r['Manifestacion_manual'] or r['Manifestacion_propuesta']
        if not w or not m or (m in work_m and work_m[m]!=w):
            raise ValueError('Una manifestación debe pertenecer a una única obra.')
        work_m[m]=w; rep_m[r['Representacion_ID']]=m
        sub = work.iloc[[i-1 for i in ri]]
        for c in BIB:
            if not sub[c].eq(r[c]).all():
                raise ValueError(f'El plan modificó el metadato original {c}.')
        if ' | '.join(sub.indice)!=r['indices_originales']:
            raise ValueError('Los índices históricos del perfil no corresponden a las filas.')
        if set(unicos(sub.Autor_norm)) != set(r['Autores'].split(' | ')):
            raise ValueError('El perfil perdió o inventó autores.')
        for i in ri:
            if i in m_row:
                raise ValueError('Una fila original aparece en varias representaciones.')
            m_row[i]=m; w_row[i]=w
        rows_m[m].extend(ri); rows_w[w].extend(ri)
        signature = (r['score_obra_diagnostico_min'],r['score_manifestacion_diagnostico_min'],r['reglas_match_diagnostico'])
        if m in score_m and score_m[m]!=signature:
            raise ValueError('La metainformación de score del grupo es inconsistente.')
        score_m[m]=signature
    if set(m_row) != set(range(1,len(work)+1)):
        raise ValueError('Se perdieron filas de entrada en el plan.')
    for dic in [rows_m,rows_w]:
        for key in dic: dic[key].sort()
    if (len(rows_m),len(rows_w))!=(561,553):
        raise ValueError('El agrupamiento difiere del plan revisado (561 manifestaciones, 553 obras).')
    return rows_m,rows_w,work_m,m_row,w_row,score_m


# ============================================================
# 4. INTERPRETAR RESOLUCIONES, SIN CREAR NUEVOS VALORES
# ============================================================
def indexar_decisiones(rev,rows_m,rows_w):
    result={}
    for r in rev.to_dict('records'):
        allowed=[x.strip() for x in r['Acciones_permitidas'].split(';')]
        if r['Decision_manual'] not in allowed:
            raise ValueError('Acción no autorizada en '+r['Caso_ID'])
        if not r['Comentario_manual'].strip():
            raise ValueError('Decisión sin justificación: '+r['Caso_ID'])
        if r['Tipo'] in {'IDENTIDAD','SIN_IDENTIFICADORES'}:
            continue
        key=(r['Tipo'],r['Nivel'],r['Objetivo_ID'],r['Campo'],r['Autor_norm'])
        if key in result:
            raise ValueError('Decisiones duplicadas para el mismo campo y ámbito.')
        if r['Nivel']=='MANIFESTACION' and r['Objetivo_ID'] not in rows_m:
            raise ValueError('Decisión referida a una manifestación inexistente.')
        if r['Nivel']=='OBRA' and r['Objetivo_ID'] not in rows_w:
            raise ValueError('Decisión referida a una obra inexistente.')
        r['donantes']=ids(r['row_ids_evidencia'])
        r['instruccion']=json.loads(r['Resolucion_JSON'])
        if r['instruccion']['accion']!=r['Decision_manual']:
            raise ValueError('Acción JSON distinta de la decisión de la fila.')
        result[key]=r
    return result


def variantes(work,rows,campo):
    groups={}
    for i in rows:
        value=work.iloc[i-1][campo]
        groups.setdefault(value,[]).append(i)
    return [{'valor':v,'row_ids':ii} for v,ii in groups.items()]


def resolver_campo(work,rows,campo,nivel,objetivo,decisiones):
    r=decisiones.get(('CAMPO',nivel,objetivo,campo,''))
    external=decisiones.get(('HALLAZGO_EXTERNO',nivel,objetivo,campo,''))
    if external:
        r=external
    vals=unicos(work.iloc[[i-1 for i in rows]][campo])
    if r:
        action=r['Decision_manual']; value=r['Valor_resuelto']; donors=r['donantes']
        if not set(donors)<=set(rows):
            raise ValueError(f"{r['Caso_ID']}: donantes fuera del contexto {nivel}.")
        if action=='MANTENER_POR_VERSION':
            return None,{'donantes':[],'metodo':action,'caso':r['Caso_ID'],'fuentes':r['Fuentes_evidencia']}
        if action=='AUTORIZAR_CORRECCION_EXTERNA':
            if (r['Caso_ID']!='C_E84681248711E1A3' or campo!='Año' or value!='2024'
                or rows!=[4540] or work.iloc[4539].Año!='2025' or not r['Fuentes_evidencia']):
                raise ValueError('Corrección externa no incluida en este checkpoint.')
        elif action=='RECHAZAR_PARAFRASIS_NO_PUBLICADA':
            if r['Caso_ID'] not in CASOS_RECHAZO_ABSTRACT or campo!='Abstract' or value!='' or not r['Fuentes_evidencia']:
                raise ValueError('Vaciado de abstract no respaldado por la revisión específica.')
        elif action=='SELECCIONAR_FRAGMENTO_PUBLICADO':
            if r['Caso_ID']!='C_9B321BA99DCD4158' or campo!='Abstract' or len(donors)!=1:
                raise ValueError('Recorte de texto no autorizado.')
            text=work.iloc[donors[0]-1][campo]; marker=' Highlights '
            if text.count(marker)!=1 or value!=text.split(marker)[0]:
                raise ValueError('El fragmento elegido no es el prefijo exacto previo a Highlights.')
        elif campo in {'ISBN','ISSN','Keywords'}:
            if action not in {'SELECCIONAR','SELECCIONAR_COMPONENTES','COMBINAR'}:
                raise ValueError('Acción inesperada sobre identificador multivaluado.')
            available={norm_id(t) if campo!='Keywords' else t.casefold()
                       for i in donors for t in tokens(work.iloc[i-1][campo])}
            selected=[norm_id(t) if campo!='Keywords' else t.casefold() for t in tokens(value)]
            if not selected or not set(selected)<=available or len(selected)!=len(set(selected)):
                raise ValueError('Componentes seleccionados inexistentes o repetidos.')
        else:
            if value not in vals or not donors or any(work.iloc[i-1][campo]!=value for i in donors):
                raise ValueError(f"{r['Caso_ID']}: no se seleccionó un valor exacto de las filas donantes.")
            if campo=='Doi' and len(vals)>1:
                if action!='CORREGIR_DOI_INTERNO' or r['Caso_ID']!='C_4DAA1F5DB665BE41':
                    raise ValueError('Dos DOI no se fusionan sin la excepción contextual revisada.')
                if {EXCEPCION_DOI.get(v,v) for v in vals}!={value}:
                    raise ValueError('Diferencia DOI fuera de la errata documentada.')
        return value,{'donantes':donors,'metodo':r['Metodo_resolucion'],'caso':r['Caso_ID'],
                      'fuentes':r['Fuentes_evidencia'],'limite':r['Limitacion']}
    if campo=='SubArea':
        if vals: raise ValueError('SubArea no vacía.')
        return '',{'donantes':[],'metodo':'SUBAREA_VACIA','caso':'','fuentes':''}
    if not vals:
        return '',{'donantes':[],'metodo':'VACIO_ORIGINAL_CONSERVADO','caso':'','fuentes':''}
    if campo in {'ISBN','ISSN','Keywords'}:
        value=union_tokens(vals,campo)
        # Identificadores distintos sin intersección requieren revisión explícita.
        if campo in {'ISBN','ISSN'} and len(vals)>1:
            sets=[{norm_id(x) for x in tokens(v)} for v in vals]
            if not any(all(s<=anchor for s in sets) for anchor in sets):
                raise ValueError(f'{objetivo}: {campo} incompatibles sin revisión.')
        return value,{'donantes':[i for i in rows if work.iloc[i-1][campo]],
                      'metodo':'UNION_COMPONENTES_EXISTENTES','caso':'','fuentes':''}
    key=norm_title if campo=='Titulo' else norm_url if campo=='URL' else lambda s:norm_text(s).casefold() if campo=='Abstract' else s
    if len({key(v) for v in vals})==1:
        value=min(vals,key=lambda x:(not x.startswith('https://'),len(x),x)) if campo=='URL' else vals[0]
        method='VALOR_EXISTENTE_EQUIVALENTE'
    elif campo=='Abstract':
        candidates=[v for v in vals if all(norm_text(other).casefold() in norm_text(v).casefold() for other in vals)]
        if not candidates:
            raise ValueError(f'{objetivo}: Abstract distinto sin dictamen.')
        value=max(candidates,key=len); method='CONTENCION_LITERAL_DEL_TEXTO_EXISTENTE'
    else:
        raise ValueError(f'{objetivo}: conflicto de {campo} sin decisión.')
    return value,{'donantes':[i for i in rows if work.iloc[i-1][campo]==value],
                  'metodo':method,'caso':'','fuentes':''}


# ============================================================
# 5. CONSOLIDAR METADATOS POR MANIFESTACIÓN Y OBRA
# ============================================================
def resolver_articulos(work,rows_m,rows_w,work_m,dec):
    articles={}; provenance={}; excepciones=[]
    for m,rr in rows_m.items():
        art={}; prov={}
        for c in BIB:
            art[c],prov[c]=resolver_campo(work,rr,c,'MANIFESTACION',m,dec)
            prov[c]['ambito']='MANIFESTACION';prov[c]['scope_rows']=rr
        if doi_url(art['URL']) and art['Doi'] and doi_url(art['URL'])!=art['Doi']:
            raise ValueError(f'{m}: contradicción DOI vs URL después de resolver.')
        articles[m]=art;provenance[m]=prov
    for w,rr in rows_w.items():
        ms=[m for m in rows_m if work_m[m]==w]
        if len(ms)<2:
            continue
        version=dec.get(('VERSION','OBRA',w,'Contenido',''))
        if not version:
            raise ValueError('Obra multimanifestación sin decisión explícita de armonización.')
        for c in ['Año','Area','Keywords','Abstract']:
            if c in {'Keywords','Abstract'} and version['Decision_manual']=='NO_ARMONIZAR_VERSIONES':
                excepciones.append((w,c,version['Caso_ID'],'CONTENIDO_POR_VERSION'))
                continue
            value,prov=resolver_campo(work,rr,c,'OBRA',w,dec)
            if value is None:
                excepciones.append((w,c,prov['caso'],'MANTENER_POR_VERSION'))
                continue
            for m in ms:
                articles[m][c]=value
                provenance[m][c]={**prov,'ambito':'OBRA','scope_rows':rr}
    return articles,provenance,excepciones


# ============================================================
# 6. AFILIACIONES DEL MISMO AUTOR, NO DE SUS COAUTORES
# ============================================================
def afiliaciones_seguras(work,rr,autor):
    rows=work.iloc[[i-1 for i in rr]]
    if not rows.Autor_norm.eq(autor).all():
        raise ValueError('Se intentó copiar afiliaciones de otro autor.')
    affsets=[];ordered=[]
    for row in rows.itertuples(index=False):
        aff=unicos([row.Afiliacion1,row.Afiliacion2])
        specific=[a for a in aff if a!=GENERICA]
        affsets.append(set(specific))
        ordered.extend(specific)
    result=unicos(ordered)
    if not result:
        if not ((rows.Afiliacion1==GENERICA)|(rows.Afiliacion2==GENERICA)).all():
            raise ValueError('No hay una afiliación UNAM documentada.')
        return [GENERICA]
    if len(result)>2 or not any(set(result)<=anchor for anchor in affsets):
        raise ValueError('Afiliaciones específicas contradictorias o sin fila que respalde la doble afiliación.')
    return result


# ============================================================
# 7. NUEVOS ÍNDICES Y FUSIÓN DE RELACIONES AUTOR-PUBLICACIÓN
# ============================================================
def producir_base(original,work,plan,rev,dec,rows_m,rows_w,work_m,score_m,articles,provenance):
    ordered=sorted(rows_m,key=lambda m:(min(rows_m[m]),m))
    newindex={m:str(i+1) for i,m in enumerate(ordered)}
    out=[];lineage=[];fieldaudit=[]
    for m in ordered:
        rr=rows_m[m];w=work_m[m];sub=work.iloc[[i-1 for i in rr]]
        authors=unicos(sub.Autor_norm)
        # No existe una incorporación de autores entre versiones aprobada en este checkpoint.
        for a in authors:
            direct=[i for i in rr if work.iloc[i-1].Autor_norm==a]
            aff=afiliaciones_seguras(work,direct,a);adonors=direct.copy();amethod='AFILIACION_RESPALDADA_MISMO_AUTOR_MANIFESTACION';acase='';asources=''
            fix=dec.get(('CORRECCION_AFILIACION','MANIFESTACION',m,'Afiliaciones',a))
            if fix:
                adonors=fix['donantes']
                if (fix['Decision_manual']!='SELECCIONAR_GENERICA_INTERNA' or
                    not set(adonors)<=set(rows_w[w]) or
                    any(work.iloc[i-1].Autor_norm!=a for i in adonors)):
                    raise ValueError('Corrección de afiliación fuera de autor/obra.')
                pair=afiliaciones_seguras(work,adonors,a)
                if pair!=[GENERICA] or fix['Valor_resuelto']!=GENERICA:
                    raise ValueError('La corrección conservadora no coincide con el donante genérico.')
                aff=pair;amethod=fix['Metodo_resolucion'];acase=fix['Caso_ID'];asources=fix['Fuentes_evidencia']
            upgrade=dec.get(('AFILIACION_GENERICA','MANIFESTACION',m,'Afiliaciones',a))
            if upgrade:
                if upgrade['Decision_manual']!='NO_PROPAGAR':
                    raise ValueError('Este checkpoint no autoriza precisar la afiliación genérica.')
                acase=upgrade['Caso_ID'];asources=upgrade['Fuentes_evidencia'];amethod='NO_PROPAGAR_AFILIACION_ESPECIFICA_NO_RESPALDADA'
            result={**articles[m],'indice':newindex[m],'Autor_norm':a,
                    'Afiliacion1':aff[0],'Afiliacion2':aff[1] if len(aff)==2 else ''}
            out.append({c:result[c] for c in CANON})
            changed=[c for c in BIB if any(work.iloc[i-1][c]!=result[c] for i in direct)]
            for c in ['Afiliacion1','Afiliacion2']:
                if any(work.iloc[i-1][c]!=result[c] for i in direct):changed.append(c)
            scope_cases=rev[rev.Objetivo_ID.isin([w,m])]
            so,sm,rules=score_m[m]
            lineage.append({'nuevo_indice':newindex[m],'Obra_ID':w,'Manifestacion_ID':m,'Autor_norm':a,
                'row_ids_originales':' | '.join(map(str,direct)),
                'indices_originales':' | '.join(original.iloc[[i-1 for i in direct]].indice),
                'fuentes_originales':' | '.join(unicos(original.iloc[[i-1 for i in direct]].Fuente_origen)),
                'Titulo':result['Titulo'],'Autores':' | '.join(authors),
                'firma_bibliografica':js({c:result[c] for c in ['Doi','ISBN','ISSN','URL']}),
                'regla_match':'PLAN_IDENTIDAD_REVISADO; '+rules,'score_obra':so,'score_manifestacion':sm,
                'campos_propagados':'; '.join(unicos(changed)),
                'conflictos':' | '.join(scope_cases.Caso_ID),
                'decision':'FUSION_RELACION_MISMO_AUTOR' if len(direct)>1 else 'RELACION_CONSERVADA',
                'filas_absorbidas':str(len(direct)-1),
                'row_ids_donantes_afiliacion':' | '.join(map(str,adonors)),
                'SHA256_entrada':SHA256_ENTRADA})
            for c in ['Autor_norm','Afiliacion1','Afiliacion2']:
                value=result[c]
                donors=direct if c=='Autor_norm' else [i for i in adonors if value and value in [work.iloc[i-1].Afiliacion1,work.iloc[i-1].Afiliacion2]]
                if value and not donors:raise ValueError('Nombre/afiliación sin donante del autor.')
                fieldaudit.append({'nuevo_indice':newindex[m],'Obra_ID':w,'Manifestacion_ID':m,'Autor_norm':a,
                    'Campo':c,'Valor_final':value,'Valores_originales':js(variantes(work,direct,c)),
                    'row_ids_donantes':' | '.join(map(str,donors)),
                    'Ambito':'AUTOR_OBRA' if fix else 'AUTOR_MANIFESTACION',
                    'Metodo':'AUTOR_NORMALIZADO_INMUTABLE' if c=='Autor_norm' else amethod,
                    'Caso_ID':'' if c=='Autor_norm' else acase,'Fuentes_evidencia':'' if c=='Autor_norm' else asources,
                    'SHA256_entrada':SHA256_ENTRADA})
        for c in BIB:
            pr=provenance[m][c];value=articles[m][c];donors=pr['donantes'];scope=pr['scope_rows']
            if not set(donors)<=set(scope):raise ValueError('Proveniencia bibliográfica fuera de ámbito.')
            if c in {'ISBN','ISSN','Keywords'}:
                available={norm_id(t) if c!='Keywords' else t.casefold() for i in donors for t in tokens(work.iloc[i-1][c])}
                if not {norm_id(t) if c!='Keywords' else t.casefold() for t in tokens(value)}<=available:
                    raise ValueError('Token nuevo no presente en donantes.')
            elif value and value not in set(work.iloc[[i-1 for i in scope]][c]):
                if pr.get('caso') not in {'C_E84681248711E1A3','C_9B321BA99DCD4158'}:
                    raise ValueError('Valor escalar inventado o sin corrección contextual autorizada.')
            fieldaudit.append({'nuevo_indice':newindex[m],'Obra_ID':w,'Manifestacion_ID':m,'Autor_norm':'',
                'Campo':c,'Valor_final':value,'Valores_originales':js(variantes(work,rr,c)),
                'row_ids_donantes':' | '.join(map(str,donors)),'Ambito':pr['ambito'],
                'Metodo':pr['metodo'],'Caso_ID':pr['caso'],'Fuentes_evidencia':pr.get('fuentes',''),
                'SHA256_entrada':SHA256_ENTRADA})
    return pd.DataFrame(out,columns=CANON),pd.DataFrame(lineage),pd.DataFrame(fieldaudit),newindex


# ============================================================
# 8. VALIDACIONES DE LA BASE FINAL Y DE SU LINAJE
# ============================================================
def validar_salida(out,lineage,fields,work,articles,rows_m,work_m,exceptions):
    if list(out.columns)!=CANON or out.shape!=(1249,14):
        raise ValueError('La salida no coincide con el cierre de 1249 relaciones y 14 columnas.')
    if out.duplicated(['indice','Autor_norm']).any():
        raise ValueError('Autor repetido dentro de un índice.')
    if set(out.Autor_norm)!=set(work.Autor_norm):
        raise ValueError('Se perdieron o inventaron nombres de autor.')
    if out.SubArea.ne('').any() or out.Afiliacion1.eq('').any():
        raise ValueError('SubArea no vacía o afiliación principal vacía.')
    if ((out.Afiliacion1==out.Afiliacion2)&out.Afiliacion2.ne('')).any():
        raise ValueError('Afiliaciones repetidas.')
    if not out.Año.isin(['','2024','2025']).all() or not out.Area.isin(AREAS).all():
        raise ValueError('Año/Area inválidos.')
    if set(out.indice)!=set(map(str,range(1,len(rows_m)+1))):
        raise ValueError('Índices finales no deterministas/consecutivos.')
    for _,g in out.groupby('indice',sort=False):
        if (g[BIB].nunique(dropna=False)!=1).any():
            raise ValueError('Metadatos distintos dentro del mismo índice.')
    for d,g in out[out.Doi.ne('')].groupby('Doi',sort=False):
        if g.indice.nunique()!=1:raise ValueError('Mismo DOI repartido innecesariamente en índices: '+d)
    for m,rr in rows_m.items():
        original_dois=set(work.iloc[[i-1 for i in rr]].Doi)-{''}
        corrected={EXCEPCION_DOI.get(d,d) for d in original_dois}
        if len(corrected)>1 or corrected and articles[m]['Doi'] not in corrected:
            raise ValueError('DOI de manifestaciones diferentes colapsados.')
    for value in out.Keywords:
        tt=[t.casefold() for t in tokens(value)]
        if len(tt)!=len(set(tt)):raise ValueError('Keywords repetidas dentro de la celda.')
    flat=[i for s in lineage.row_ids_originales for i in ids(s)]
    if len(flat)!=len(work) or set(flat)!=set(range(1,len(work)+1)):
        raise ValueError('Linaje incompleto o duplicado.')
    if sum(int(x) for x in lineage.filas_absorbidas)!=len(work)-len(out):
        raise ValueError('Las absorciones no concuerdan con las filas finales.')
    if len(fields)!=len(rows_m)*len(BIB)+len(out)*3:
        raise ValueError('Auditoría de campos incompleta.')
    for w in set(work_m.values()):
        ms=[m for m in rows_m if work_m[m]==w]
        if len({articles[m]['Area'] for m in ms})!=1:raise ValueError('Área inconsistente en la obra.')
        yy={articles[m]['Año'] for m in ms}-{''}
        if len(yy)>1 and not any(x[0]==w and x[1]=='Año' for x in exceptions):
            raise ValueError('Años de obra contradictorios sin excepción explícita.')
    validar_identificadores(out)


# ============================================================
# 9. RESUMEN Y ESCRITURA SEGURA CON RELECTURA
# ============================================================
def hacer_resumen(original,out,rev,lineage,fields,plan,exceptions):
    summary=[
        ('GENERAL','Filas_entrada',len(original),'Sin modificaciones a la entrada.'),
        ('GENERAL','Obras',553,'Etiquetas del plan de identidades revisado.'),
        ('GENERAL','Manifestaciones',561,'Un nuevo indice por manifestación.'),
        ('GENERAL','Relaciones_finales',len(out),'Una fila por manifestación + autor UNAM.'),
        ('GENERAL','Filas_absorbidas',len(original)-len(out),'Sin eliminar relaciones únicas ni personas.'),
        ('GENERAL','Autores_unicos',out.Autor_norm.nunique(),'Nombres inmutables; no incorpora nuevos autores.'),
        ('GENERAL','Columnas_finales',len(out.columns),'Fuente_origen eliminado del canónico.'),
        ('REVISION','Pendientes_originales_resueltos',476,'Se preservan también los 149 cierres anteriores.'),
        ('REVISION','Correccion_contextual_adicional',1,'Gilde: conservar UNAM genérica ante la especificidad contradictoria de la misma obra.'),
        ('REVISION','Decisiones_totales',len(rev),'149 previas + 476 revisadas + 1 hallazgo contextual.'),
        ('REVISION','Decisiones_sin_resolver',int(rev.Decision_manual.eq('').sum()),'0 no equivale a 100% completitud de datos.'),
        ('REVISION','Abstracts_rechazados_por_parafrasis',5,'Vacíos documentados; no se copió el abstract externo.'),
        ('REVISION','Abstracts_solo_prefijo_cientifico',1,'Recorte exacto antes de Highlights, corroborado editorialmente.'),
        ('REVISION','Correcciones_externas_Año',1,'IMAV: 2025 -> 2024. No se completa URL/DOI.'),
        ('REVISION','Correcciones_contextuales_DOI',1,'373426 -> 3734260, solo la nota expresamente identificada.'),
        ('REVISION','Autores_anadidos_entre_manifestaciones',0,'No se igualaron conjuntos por suposición.'),
        ('LINAJE','Relaciones_auditadas',len(lineage),'Cada fila original está representada una vez.'),
        ('LINAJE','Campos_auditados',len(fields),'Diez metadatos por manifestación y tres campos por relación.'),
        ('HUELLA','SHA256_entrada',SHA256_ENTRADA,''),
        ('HUELLA','SHA256_plan',SHA256_PLAN,''),
        ('HUELLA','SHA256_decisiones',SHA256_DECISIONES,''),
        ('REGLA','Orden_indices','PRIMERA_APARICION_ORIGINAL','Desempate por identificador estable de manifestación.'),
        ('REGLA','Uso_de_scores','SOLO_AUDITORIA_DIAGNOSTICA','El 07 no decide por threshold ni reabre identidades.'),
        ('REGLA','Internet_durante_ejecucion','NO','La revisión utilizó fuentes externas indicadas en los CSV.'),
    ]
    for c in ['Año','ISBN','ISSN','Doi','URL','Keywords','Abstract']:
        summary.append(('COMPLETITUD',c+'_vacios_final',int(out[c].eq('').sum()),'Conteo por relación final, no comparable directamente con filas de entrada.'))
    for action,n in rev.Decision_manual.value_counts().items():
        summary.append(('DECISION',action,int(n),''))
    for _,r in rev[rev.Limitacion.ne('')].iterrows():
        summary.append(('LIMITACION',r.Caso_ID,r.Campo,r.Limitacion))
    for w,c,cid,rule in exceptions:
        summary.append(('EXCEPCION',w,c,rule+'; '+cid))
    return pd.DataFrame(summary,columns=['Tipo','Concepto','Valor','Comentario']).astype(str)


def guardar_resultados(tablas,carpeta,actualizar):
    carpeta.mkdir(parents=True,exist_ok=True)
    temporales=[]
    try:
        # Prevalidar TODOS los destinos antes de reemplazar cualquiera.
        for name,df in tablas.items():
            target=carpeta/name; esperado=df.fillna('').astype(str).reset_index(drop=True)
            if target.exists() and not actualizar:
                actual=leer_csv(target)
                if not actual.equals(esperado):
                    raise FileExistsError(f'{name}: salida distinta ya existente. Resguardar en Git y autorizar actualizar_archivos=True.')
        for name,df in tablas.items():
            target=carpeta/name; temp=target.with_name(target.name+'.__tmp__')
            expected=df.fillna('').astype(str).reset_index(drop=True)
            expected.to_csv(temp,index=False,encoding='utf-8-sig',quoting=csv.QUOTE_ALL,lineterminator='\n')
            if not leer_csv(temp).equals(expected):
                raise ValueError('Fallo de relectura: '+name)
            temporales.append((temp,target,expected))
        for temp,target,expected in temporales:
            temp.replace(target)
            if not leer_csv(target).equals(expected):raise ValueError('Salida distinta después de guardar: '+target.name)
    finally:
        for temp,_,_ in temporales:
            if temp.exists():temp.unlink()


# ============================================================
# 10. EJECUTAR EL CIERRE REVISADO
# ============================================================
raiz=localizar_raiz()
original,trabajo,plan,revisiones,carpeta_salida,insumos,huellas=cargar_checkpoint(raiz)
rows_m,rows_w,work_m,m_row,w_row,scores=organizar_plan(trabajo,plan)
decisiones=indexar_decisiones(revisiones,rows_m,rows_w)
articulos,proveniencia,excepciones=resolver_articulos(trabajo,rows_m,rows_w,work_m,decisiones)
salida,linaje,auditoria_campos,nuevos_indices=producir_base(
    original,trabajo,plan,revisiones,decisiones,rows_m,rows_w,work_m,scores,articulos,proveniencia)
validar_salida(salida,linaje,auditoria_campos,trabajo,articulos,rows_m,work_m,excepciones)
resumen=hacer_resumen(original,salida,revisiones,linaje,auditoria_campos,plan,excepciones)
for path,h in zip(insumos,huellas):
    if sha256(path)!=h:raise ValueError('Un insumo fue modificado durante la ejecución.')
guardar_resultados({
    'autores_unam_deduplicados.csv':salida,
    'auditoria_fusion.csv':linaje,
    'auditoria_campos_fusion.csv':auditoria_campos,
    'resumen_fusion.csv':resumen,
},carpeta_salida,actualizar_archivos)
for path,h in zip(insumos,huellas):
    if sha256(path)!=h:raise ValueError('Un insumo cambió al guardar resultados.')
print('=== CIERRE 07 ===')
print('Filas originales:',len(original))
print('Obras:',len(rows_w),'| Manifestaciones:',len(rows_m))
print('Relaciones finales:',len(salida),'| Columnas:',len(salida.columns))
print('Filas repetidas absorbidas:',len(original)-len(salida))
print('Decisiones pendientes:',int(revisiones.Decision_manual.eq('').sum()))
print('Abstracts rechazados por no corresponder al texto publicado: 5 manifestaciones')
print('Entrada, plan y decisiones intactos: Sí')
print('Relectura exacta de todas las salidas: Sí')
print('Salida:',(carpeta_salida/'autores_unam_deduplicados.csv').relative_to(raiz))
print('SHA256 salida:',sha256(carpeta_salida/'autores_unam_deduplicados.csv'))
print('Consultar resumen_fusion.csv: cierre de conflictos NO significa completitud de todos los campos.')


=== CIERRE 07 ===
Filas originales: 5106
Obras: 553 | Manifestaciones: 561
Relaciones finales: 1249 | Columnas: 14
Filas repetidas absorbidas: 3857
Decisiones pendientes: 0
Abstracts rechazados por no corresponder al texto publicado: 5 manifestaciones
Entrada, plan y decisiones intactos: Sí
Relectura exacta de todas las salidas: Sí
Salida: 04_Limpieza\04_deduplicacion\autores_unam_deduplicados.csv
SHA256 salida: 1a4d5d19d289d1a0ff5fd8572e9edb8f1f414227f7f13fd746c2028907bd4409
Consultar resumen_fusion.csv: cierre de conflictos NO significa completitud de todos los campos.
